In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
import re

spark.sql("USE CATALOG 02_silver")
spark.sql("USE SCHEMA default")

def clean_column(name):
    name = re.sub(r'[^a-zA-Z0-9]', '', name)
    return name.lower()

**Function for cleaning dates**

In [0]:
from pyspark.sql.functions import expr, coalesce, to_date
def parse_mixed_date(colname):
    clean_col = f"trim({colname})"
    return to_date(
        coalesce(
            expr(f"try_to_timestamp({clean_col}, 'yyyyMMdd')"),
            expr(f"try_to_timestamp({clean_col}, 'dd-MM-yyyy')"),
            expr(f"try_to_timestamp({clean_col}, 'MM-dd-yyyy')"),
            expr(f"try_to_timestamp({clean_col}, 'MM/dd/yyyy')"),
            expr(f"try_to_timestamp({clean_col}, 'dd-MMM-yy')"),
            expr(f"try_to_timestamp({clean_col}, 'MMM dd, yyyy')"),
            expr(f"try_to_timestamp({clean_col}, 'yyyy.MM.dd')"),
            expr(f"try_to_timestamp({clean_col}, 'yyyy-MM-dd')")
        )
    )

**Cleaning customer table**

In [0]:
df = spark.read.table("01_bronze.default.crm_cust_base_raw")
df = df.select([col(c).alias(clean_column(c)) for c in df.columns])

df = df.withColumn("parsed_date", parse_mixed_date("joineddate"))

df = df.withColumn("raw_phone", regexp_replace(col("contactinfo"), "[^0-9]", ""))

df = df.withColumn(
    "phone_10",
    when(length(col("raw_phone")) >= 10, substring(col("raw_phone"), -10, 10))
)

df = df.withColumn(
    "is_valid_mobile",
    when(col("phone_10").rlike("^[6-9][0-9]{9}$"), 1).otherwise(0)
)

In [0]:
window = Window.partitionBy("custrefid").orderBy(
    col("is_valid_mobile").desc(),
    length(col("raw_phone")).desc()
)

df = df.withColumn("rank", row_number().over(window))

df_customers = df.filter(col("rank") == 1).drop("rank", "raw_phone", "is_valid_mobile")

df_customers = df_customers.withColumn(
    "phone_10",
    when(col("phone_10").isNull(), lit("0000000000"))
    .otherwise(col("phone_10"))
)

df_customers = df_customers.withColumn(
    "phone_number",
    col("phone_10")
)

In [0]:
df_customers = df_customers.select(
    col("custrefid").alias("customer_id"),
    initcap(trim(regexp_replace(col("fullname"), "[^a-zA-Z\\s]", ""))).alias("customer_name"),
    col("phone_number"),
    to_date(col("parsed_date")).alias("joined_date"),
    col("gendercode").alias("gender")
)
spark.sql("DROP TABLE IF EXISTS 02_silver.default.customers")

df_customers.write.format("delta").mode("overwrite").saveAsTable("02_silver.default.customers")


**Cleaning the sales table**

In [0]:
df = spark.read.table("01_bronze.default.raw_sales_dtl")

df = df.select([col(c).alias(clean_column(c)) for c in df.columns])
df_sales = df.select(
    col("trxnid").alias("transaction_id"),
    col("custid99").alias("customer_id"),
    col("storelocid").alias("store_id"),
    col("prodcodeid").alias("product_id"),
    
    parse_mixed_date("dateref").alias("transaction_date"),
    col("qtysold").cast("int").alias("quantity"),
    regexp_replace(col("unitprice"), "[^0-9.]", "").cast("double").alias("selling_price")
)
 
df_sales = df_sales.withColumn("ingested_at", current_timestamp())
df_sales = df_sales.withColumn(
    "customer_id",
    when(
        col("customer_id").isNull() |
        (trim(col("customer_id")) == "") |
        (lower(trim(col("customer_id"))) == "null"),
        "unknown"
    ).otherwise(col("customer_id"))
)

print("Row count:", df_sales.count())
df_sales.printSchema()
spark.sql("DROP TABLE IF EXISTS 02_silver.default.sales")

df_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("02_silver.default.sales")

**Cleaning the inventory table**

In [0]:
df = spark.read.table("01_bronze.default.inv_levels_raw")

df = df.select([col(c).alias(clean_column(c)) for c in df.columns])
df = df.withColumn("parsed_date", parse_mixed_date("lastauditdt"))
df_inventory = df.select(
    col("skuid").alias("product_id"),
    col("stidref").alias("store_id"),
    col("stockonhand").cast("int").alias("stock_qty"),
    col("parsed_date").alias("last_audit_date")
)

df_inventory = df_inventory.fillna({
    "stock_qty": 0
})


df_inventory = df_inventory.dropDuplicates()
spark.sql("DROP TABLE IF EXISTS inventory")

df_inventory.write.format("delta").mode("overwrite").saveAsTable("inventory")

**Cleaning the returns table**

In [0]:
df = spark.read.table("01_bronze.default.rtn_trans_logs")

df = df.select([col(c).alias(clean_column(c)) for c in df.columns])

df_returns = df.select(
    col("orgnltrxnid").alias("transaction_id"),
    col("rtnidno").alias("return_id"),
    
    parse_mixed_date("rtndatestring").alias("return_date").alias("return_date"),
    
    col("rsncode").alias("return_reason")
)

df_returns = df_returns.fillna({
    "return_reason": "unknown"
})

df_returns = df_returns.dropDuplicates(["return_id"])

df_returns = df_returns.withColumn("ingested_at", current_timestamp())

spark.sql("DROP TABLE IF EXISTS returns")

df_returns.write.format("delta").mode("overwrite").saveAsTable("returns")